In [14]:
import json
import os
from transformers import AutoTokenizer

In [ ]:
MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO" 

if not tokenizer.is_fast:
    raise RuntimeError("Fast tokenizer required")

In [16]:
INPUT_PATH = "data/literature/SNCA_A53T_corpus.json"
OUTPUT_PATH = "data/literature/chunked_corpus.json"

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"❌ File not found: {INPUT_PATH}")

with open(INPUT_PATH, "r") as f:
    papers = json.load(f)

print(f"📚 Loaded papers: {len(papers)}")

📚 Loaded papers: 100


In [17]:
def chunk_by_tokens_offsets(text, tokenizer, chunk_size=400, overlap=80):
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(input_ids), step):
        window_offsets = offsets[i : i + chunk_size]

        if not window_offsets:
            continue

        start_char = window_offsets[0][0]
        end_char = window_offsets[-1][1]

        chunk_text = text[start_char:end_char]
        chunks.append(chunk_text)

        if i + chunk_size >= len(input_ids):
            break

    return chunks

In [18]:
chunked_data = []

for paper in papers:
    pmid = paper.get("pmid")
    title = paper.get("title", "")
    abstract = paper.get("abstract", "")

    if not abstract:
        continue

    # ✅ Normalize text (NEW FIX)
    title = title.replace("\n", " ").strip()
    abstract = abstract.replace("\n", " ").strip()

    # ✅ Chunking
    chunks = chunk_by_tokens_offsets(abstract, tokenizer)

    # ✅ Add chunk IDs (NEW FIX)
    for i, chunk in enumerate(chunks):
        chunked_data.append({
            "chunk_id": f"{pmid}_{i}",   # UNIQUE ID
            "pmid": pmid,
            "title": title,
            "chunk": chunk
        })

print(f"🧩 Total chunks created: {len(chunked_data)}")

Token indices sequence length is longer than the specified maximum sequence length for this model (534 > 512). Running this sequence through the model will result in indexing errors


🧩 Total chunks created: 120


In [19]:
with open(OUTPUT_PATH, "w") as f:
    json.dump(chunked_data, f, indent=4, ensure_ascii=False)

print(f"✅ Saved to {OUTPUT_PATH}")

✅ Saved to data/literature/chunked_corpus.json


In [20]:
print("Total chunks:", len(chunked_data))
print("Sample chunk length:", len(chunked_data[0]["chunk"].split()))
print(chunked_data[0])

Total chunks: 120
Sample chunk length: 171
{'chunk_id': '41002401_0', 'pmid': '41002401', 'title': 'Modeling Synucleinopathy Using hESC-Derived Cerebral Organoids.', 'chunk': 'Animal and cellular models harboring SNCA gene mutations have been instrumental in synucleinopathy, but faithful human brain models remain limited. Here, we report the development of a human cerebral organoid (CO) model of synucleinopathy carrying the Ala53Thr mutation in SNCA ( SNCA A53T ). Using a human embryonic stem cell (hESC) line overexpressing SNCA A53T (A53T hESC line), we generated COs (A53T COs) that recapitulate hallmark features of synucleinopathy. These A53T COs exhibited elevated α-synuclein (α-Syn) expression, the increased phosphorylation of α-Syn, and Lewy body-like aggregations. Notably, we also observed the increased expression of phosphorylated tau and neurofibrillary tangle-like silver deposits, although amyloid β expression and accumulation remained unchanged. To evaluate the utility of thi

In [21]:
avg_len = sum(len(tokenizer.encode(c["chunk"])) for c in chunked_data) / len(chunked_data)
print(f"Avg tokens per chunk: {avg_len:.2f}")

print("Empty chunks present:",
      any(len(c["chunk"].strip()) == 0 for c in chunked_data))

Avg tokens per chunk: 280.26
Empty chunks present: False
